# Edit distance & string algorithms — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def softmax(x, axis=-1):
    x=np.asarray(x, dtype=float); x=x-x.max(axis=axis, keepdims=True); e=np.exp(x); return e/e.sum(axis=axis, keepdims=True)
def sigmoid(x): return 1/(1+np.exp(-np.asarray(x, dtype=float)))
def normalize(v):
    v=np.asarray(v, dtype=float); n=np.linalg.norm(v); return v/(n if n else 1)
def edit_distance(a,b):
    D=np.zeros((len(a)+1,len(b)+1), dtype=int); D[:,0]=np.arange(len(a)+1); D[0,:]=np.arange(len(b)+1)
    for i in range(1,len(a)+1):
        for j in range(1,len(b)+1):
            D[i,j]=min(D[i-1,j]+1, D[i,j-1]+1, D[i-1,j-1]+(a[i-1]!=b[j-1]))
    return D
print('setup ok')

## Dynamic programming for minimum string edits
These cells build the Levenshtein table, contrast edit operations, show backtracking, expose a weighted substitution knob, and visualize approximate matching.

In [ ]:
a,b='cat','cut'; D=edit_distance(a,b)
plt.figure(figsize=(4,3)); plt.imshow(D, cmap='Blues'); plt.colorbar(); plt.xticks(range(len(b)+1),['']+list(b)); plt.yticks(range(len(a)+1),['']+list(a)); plt.title('DP table for cat -> cut')
assert D[-1,-1]==1

In [ ]:
pairs=[('cat','cats'),('cat','cut'),('cat','at')]
dists=[edit_distance(a,b)[-1,-1] for a,b in pairs]
plt.figure(figsize=(6,3)); plt.bar([f'{a}->{b}' for a,b in pairs],dists); plt.title('Insert, substitute, delete each cost one')
assert dists==[1,1,1]

In [ ]:
a,b='kitten','sitting'; D=edit_distance(a,b)
plt.figure(figsize=(6,3)); plt.imshow(D, cmap='magma'); plt.colorbar(); plt.title('kitten -> sitting has distance 3')
assert D[-1,-1]==3

In [ ]:
def weighted(a,b,sub=2):
    D=np.zeros((len(a)+1,len(b)+1), dtype=int); D[:,0]=np.arange(len(a)+1); D[0,:]=np.arange(len(b)+1)
    for i in range(1,len(a)+1):
        for j in range(1,len(b)+1): D[i,j]=min(D[i-1,j]+1,D[i,j-1]+1,D[i-1,j-1]+(0 if a[i-1]==b[j-1] else sub))
    return D[-1,-1]
subs=[1,2,3]; vals=[weighted('cat','cut',s) for s in subs]
plt.figure(figsize=(6,3)); plt.plot(subs, vals, '-o'); plt.xlabel('substitution cost'); plt.ylabel('distance'); plt.title('Cost design changes the path')
assert vals==[1,2,2]

In [ ]:
query='color'; words=['colour','colon','cold','caller','color']; d=[edit_distance(query,w)[-1,-1] for w in words]
plt.figure(figsize=(6,3)); plt.bar(words,d); plt.axhline(2,ls='--',c='r'); plt.title('Approximate retrieval by edit radius')
assert words[int(np.argmin(d))]=='color' and sum(x<=2 for x in d)==4